# bhavpy quickstart

Indian market data from NSE/BSE public archives. First run downloads one archive file per trading day for the range (whole market) into a local cache; after that everything is instant and offline.

*Not affiliated with NSE/BSE. Personal/research use; follow the exchanges' terms. Not investment advice.*

In [1]:
# pip install "bhavpy[plot]"
import bhavpy as bp
import pandas as pd
import matplotlib.pyplot as plt
bp.__version__

'0.1.0'

## 1. Raw vs adjusted prices
RELIANCE had 1:1 bonuses in Sep 2017 and Oct 2024. The raw close halves on each ex-date; the adjusted close does not.

In [2]:
df = bp.history("RELIANCE", "2017-01-01", "2025-12-31")
ax = df[["close", "adj_close"]].plot(figsize=(10, 4), title="RELIANCE: raw vs split/bonus-adjusted")

In [3]:
bp.factors("RELIANCE", "2017-01-01", "2025-12-31", adjust="total")[
    ["ex_date", "kind", "ratio", "amount", "price_factor", "cumulative_factor", "check"]
].tail(10)

,ex_date,kind,ratio,amount,price_factor,cumulative_factor,check
3,2019-08-02 00:00:00+05:30,dividend,,6.5,0.994493,0.437956,ok
4,2020-05-13 00:00:00+05:30,rights,1:15,1257.0,0.990610,0.440381,ok
5,2020-07-02 00:00:00+05:30,dividend,,6.5,0.996259,0.444556,ok
6,2021-06-11 00:00:00+05:30,dividend,,7.0,0.996793,0.446225,ok
7,2022-08-18 00:00:00+05:30,dividend,,8.0,0.996998,0.447661,ok
8,2023-07-20 00:00:00+05:30,demerger,,NaN,0.907859,0.449009,estimated
9,2023-08-21 00:00:00+05:30,dividend,,9.0,0.996480,0.494579,ok
10,2024-08-19 00:00:00+05:30,dividend,,10.0,0.996618,0.496326,ok
11,2024-10-28 00:00:00+05:30,bonus,1:1,NaN,0.500000,0.498011,ok
12,2025-08-14 00:00:00+05:30,dividend,,5.5,0.996022,0.996022,ok


## 2. Old tickers still work
`ZOMATO` was renamed `ETERNAL` on 2025-04-09; the history is stitched.

In [4]:
bp.symbol_chain("ZOMATO")

[Segment(symbol='ZOMATO', start=None, end=datetime.date(2025, 4, 9)),
 Segment(symbol='ETERNAL', start=datetime.date(2025, 4, 9), end=None)]

In [5]:
bp.history("ZOMATO", "2025-04-01", "2025-04-15", adjust=None)[["symbol", "close"]]

,symbol,close
date,,
2025-04-01 00:00:00+05:30,ZOMATO,202.01
2025-04-02 00:00:00+05:30,ZOMATO,212.03
2025-04-03 00:00:00+05:30,ZOMATO,210.90
2025-04-04 00:00:00+05:30,ZOMATO,210.53
2025-04-07 00:00:00+05:30,ZOMATO,209.63
2025-04-08 00:00:00+05:30,ZOMATO,215.19
2025-04-09 00:00:00+05:30,ETERNAL,211.39
2025-04-11 00:00:00+05:30,ETERNAL,217.20
2025-04-15 00:00:00+05:30,ETERNAL,222.23


## 3. Index history and point-in-time constituents

In [6]:
nifty = bp.history("NIFTY 50", "2015-01-01", "2025-12-31")
nifty[["close", "pe"]].plot(subplots=True, figsize=(10, 5));

In [7]:
then = set(bp.constituents("NIFTY 50", as_of="2018-01-01")["symbol"])
now = set(bp.constituents("NIFTY 50")["symbol"])
print("left since 2018:", sorted(then - now))
print("joined since 2018:", sorted(now - then))

left since 2018: ['AMBUJACEM', 'AUROPHARMA', 'BOSCHLTD', 'BPCL', 'GAIL', 'HDFC', 'HEROMOTOCO', 'HINDPETRO', 'INDUSINDBK', 'INDUSTOWER', 'IOC', 'LUPIN', 'SAMMAANCAP', 'UPL', 'VEDL', 'WIPRO', 'YESBANK', 'ZEEL']
joined since 2018: ['ADANIENT', 'APOLLOHOSP', 'BAJAJFINSV', 'BEL', 'BSE', 'ETERNAL', 'GRASIM', 'HDFCLIFE', 'INDIGO', 'JIOFIN', 'JSWSTEEL', 'MAXHEALTH', 'NESTLEIND', 'SBILIFE', 'SHRIRAMFIN', 'TATACONSUM', 'TITAN', 'TRENT']


## 4. Moving-average backtest on adjusted prices

In [8]:
import numpy as np
px = bp.history("TCS", "2015-01-01", "2025-12-31", adjust="total")["adj_close"]
pos = (px.rolling(50).mean() > px.rolling(200).mean()).astype(int).shift(1).fillna(0)
ret = px.pct_change().fillna(0)
strat = pos * ret - 0.001 * pos.diff().abs().fillna(0)
pd.DataFrame({"buy & hold": (1 + ret).cumprod(), "SMA 50/200": (1 + strat).cumprod()}).plot(figsize=(10, 4), logy=True);

## 5. Calendar, F&O, flows

In [9]:
bp.holidays(2026).head()

,description,weekday,muhurat_session,source
date,,,,
2026-01-15 00:00:00+05:30,Municipal Corporation Election - Maharashtra,Thursday,False,nse_api
2026-01-26 00:00:00+05:30,Republic Day,Monday,False,nse_api
2026-03-03 00:00:00+05:30,Holi,Tuesday,False,nse_api
2026-03-26 00:00:00+05:30,Shri Ram Navami,Thursday,False,nse_api
2026-03-31 00:00:00+05:30,Shri Mahavir Jayanti,Tuesday,False,nse_api


In [10]:
bp.lot_sizes().query("symbol in ['NIFTY', 'BANKNIFTY']").head()

,symbol,underlying,kind,month,lot_size
0,NIFTY,NIFTY 50,index,2026-10,65
1,NIFTY,NIFTY 50,index,2026-11,65
2,NIFTY,NIFTY 50,index,2026-12,65
3,NIFTY,NIFTY 50,index,2027-03,65
4,NIFTY,NIFTY 50,index,2027-06,65


In [11]:
bp.fii_dii()

,category,buy,sell,net
date,,,,
2026-10-21 00:00:00+05:30,DII,14931.57,13317.25,1614.32
2026-10-21 00:00:00+05:30,FII,17003.90,15981.58,1022.32


## 6. Live option chain → IV smile
(The option chain is the one dataset read from NSE's live site.)

In [12]:
chain = bp.option_chain("NIFTY")
spot = chain.attrs["underlying_price"]
otm = chain[((chain.option_type == "PE") & (chain.strike < spot)) | ((chain.option_type == "CE") & (chain.strike >= spot))]
otm = otm[(otm.iv > 0) & (otm.strike.between(spot * 0.9, spot * 1.1))]
ax = otm.plot(x="strike", y="iv", style="o-", title=f"NIFTY IV smile @ {chain.attrs['timestamp']}", figsize=(8, 4))
ax.axvline(spot, ls="--", c="grey");

In [13]:
from bhavpy import cache
cache.info()["size_mb"], cache.path()

(672.05,
 PosixPath('/tmp/claude-0/-home-user/cf849850-dc75-53b1-b4ab-eb54fcfc9e96/scratchpad/cache2/bhavpy.sqlite'))